# 16 — Sample + export to C (phase 4)

1. **PyTorch** `generate()` below (like notebook 9).
2. **Export** weights to a binary checkpoint.
3. **C** greedy sample with `train_v2_tiny -sample -ckpt ...`.


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text
from llmc.deepseek_v2 import DeepSeekV2, DeepSeekV2Config
from llmc.sample import generate

text = load_text("data/tiny_shakespeare.txt")
tok = CharTokenizer.from_text(text)
model = DeepSeekV2(DeepSeekV2Config.tiny(tok.vocab_size, 64))
model.eval()

prompt = "ROMEO:"
ctx = tok.encode_tensor(prompt).unsqueeze(0)
out = generate(model, ctx, max_new_tokens=120, temperature=0.9, top_k=40)
print("--- PyTorch sample ---")
print(tok.decode(out.squeeze().tolist()))


### Export for C (match `train_v2_tiny.c` tiny config)


In [ ]:
# Run once after training (or uses a short train inside the script)
import subprocess
subprocess.run([
    "python3", "scripts/export_v2_tiny.py",
    "--match-train-c", "--train-steps", "30",
    "-o", "checkpoints/v2_tiny.bin",
], check=False)


### C sample

```bash
cd c && ./bin/train_v2_tiny -sample -ckpt ../checkpoints/v2_tiny.bin
```

Greedy decoding — train longer in notebook 15 for better text.


In [ ]:
import shutil, subprocess
from pathlib import Path
ckpt = Path("checkpoints/v2_tiny.bin")
if ckpt.exists() and shutil.which("make"):
    subprocess.run(["make", "-s", "bin/train_v2_tiny"], cwd="c")
    r = subprocess.run(["./bin/train_v2_tiny", "-sample", "-ckpt", f"../{ckpt}"], cwd="c", capture_output=True, text=True)
    print(r.stdout)


## Phase 5 (later)

Full backward through MLA + MoE in C (see `vendor/llm.c/train_gpt2.c` backward passes).

Until then: **train in PyTorch**, **infer/sample in C** via export.
